# Phase 1 — Qwen3-4B baseline on Kaggle T4

Attach this repository as a Kaggle Dataset named `vietlegal-llm`, enable a T4 GPU and Internet, then run all cells. The notebook never logs chain-of-thought.

In [ ]:
import shutil
from pathlib import Path

source = Path('/kaggle/input/vietlegal-llm')
project = Path('/kaggle/working/vietlegal-llm')
if project.exists():
    shutil.rmtree(project)
shutil.copytree(source, project)
%cd /kaggle/working/vietlegal-llm
!pip -q install uv
!uv sync --frozen --extra baseline

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
!uv run --frozen vietlegal-baseline validate

In [ ]:
import json

!uv run --frozen vietlegal-baseline smoke --output artifacts/kaggle-phase1-smoke
!uv run --frozen vietlegal-baseline run --output artifacts/kaggle-phase1-full
!uv run --frozen vietlegal-baseline report --output artifacts/kaggle-phase1-full
repeat_a = 'artifacts/kaggle-phase1-repeat-a'
repeat_b = 'artifacts/kaggle-phase1-repeat-b'
!uv run --frozen vietlegal-baseline run --split test --limit 10 --output {repeat_a}
!uv run --frozen vietlegal-baseline run --split test --limit 10 --output {repeat_b}
def outputs(directory):
    path = Path(directory) / 'predictions.jsonl'
    return [json.loads(line)['raw_output'] for line in path.read_text().splitlines()]
a = outputs(repeat_a)
b = outputs(repeat_b)
assert a == b, 'Greedy repeatability check failed'
!tar -czf /kaggle/working/vietlegal-phase1-artifacts.tar.gz artifacts/kaggle-phase1-full
!sha256sum /kaggle/working/vietlegal-phase1-artifacts.tar.gz